# Reproduce van der Drift & O'Gorman (2025)
*Dependence of convective precipitation extremes on near-surface relative humidity* (arXiv:2412.16306).
Table 1, Fig. 1, Fig. 2 and Fig. 3 from the stage-2 runs of one experiment, with the paper's numbers printed alongside.

Needs, per run `<EXPERIMENT>/stage2_rv<rv>/`: `OUT_STAT/*.nc` (converted STAT), `OUT_2D/*.2Dbin`, `OUT_3D/*.bin3D`
(3-hourly over days 30-60). Only the config cell below must be run first; every other cell imports what it needs.
The expensive step (3D condensation + 100 block-bootstrap resamples per run) is cached in `<EXPERIMENT>/analysis_cache/`; delete a pickle to recompute.

Units: SAM 2D `Prec` is mm/day; $P_e$, $C_e$ here are mm/hr as in the paper. "Near-surface" = lowest model level (25 m here, 40 m in the paper), so near-surface T, RH, q_v differ slightly from Table 1 by construction.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG (the only cell the others depend on) ----
from pathlib import Path
EXPERIMENT = Path('/scratch/ttg2015/Torch/ExtremeHeat/SCM/SAM_HEAT_runs/CHANGE_ME')   # holds stage2_rv<rv>/
RV_LIST = [0, 200, 500, 1000, 2000]     # evaporative resistances r_v (s/m)
LAST_DAYS = 30                          # analysis period: last 30 days
Q = 99.9                                # extreme percentile
BLOCK = 8                               # bootstrap block: 8 x 8 gridpoints x 1 snapshot
N_BOOT = 100                            # bootstrap resamples

## Analyse every run (cached)

In [ ]:
from samheat.analysis.repro import load_summaries
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)
for s in S:
    print(f"rv={s['rv']:6g}  snapshots={s['n_snapshots']}  P_e={s['Pe']:.2f} mm/hr  C_e={s['Ce']:.1f} mm/hr  LCL={s['zlcl']:.0f} m")

## Table 1 - horizontal- and time-mean state

In [ ]:
import numpy as np, pandas as pd
from samheat.analysis.repro import load_summaries
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)

PAPER = {  # rv: (Ts, near-surface T, RH, qv, P)
    0:    (300.0, 296.9, 75.2, 14.0, 2.8),
    200:  (301.9, 297.7, 68.5, 13.4, 2.4),
    500:  (303.9, 298.7, 61.3, 12.8, 2.1),
    1000: (306.0, 299.9, 53.1, 11.9, 1.6),
    2000: (308.3, 301.4, 44.3, 10.9, 1.2)}
cols = ['Ts (K)', 'T_sfc (K)', 'RH (%)', 'qv (g/kg)', 'P (mm/day)']
rows = []
for s in S:
    t = s['table1']
    rows.append([s['rv'], *[round(t[k], 2) for k in ('Ts', 'T_sfc', 'RH', 'qv', 'P')]])
mine = pd.DataFrame(rows, columns=['rv'] + cols).set_index('rv')
paper = pd.DataFrame({rv: v for rv, v in PAPER.items() if rv in mine.index}, index=cols).T
paper.index.name = 'rv'
print('SAM_HEAT (lowest level = 25 m)'); display(mine)
print("Paper Table 1 (lowest level = 40 m)"); display(paper)
print('Difference (SAM_HEAT - paper)'); display((mine - paper).round(2))

## Fig. 1 - mean RH(z), potential temperature(z) and LCL

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.repro import load_summaries
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)
colors = ['#1a5292', '#5fa5dd', '#446400', '#d9863b', '#783c25']

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.5), constrained_layout=True)
for i, s in enumerate(S):
    pr, c = s['profiles'], colors[i % len(colors)]
    z = pr['z']
    a.plot(pr['RELH'][z < 11e3], z[z < 11e3] / 1e3, c=c, label=f"{s['rv']:g} s/m")
    a.scatter(89.5, s['zlcl'] / 1e3, c=c, marker='>', s=15)
    m = z < 3.2e3
    b.plot(pr['THETA'][m], z[m], c=c, label=f"{s['rv']:g} s/m")
    b.scatter(306.7, s['zlcl'], c=c, marker='>', s=15)
a.set_xlabel('Relative Humidity (%)'); a.set_ylabel('Height (km)'); a.set_xlim(35, 91); a.set_ylim(-.1, 11)
b.set_xlabel('Potential Temperature (K)'); b.set_ylabel('Height (m)'); b.set_ylim(-20, 3200)
b.legend(title='LCLs: triangles', frameon=False, loc='center right')
for ax in (a, b): ax.spines[['top', 'right']].set_visible(False)
a.set_title('(a)', loc='left', fontweight='bold'); b.set_title('(b)', loc='left', fontweight='bold')
plt.show()
print('LCL (m):', {s['rv']: round(s['zlcl']) for s in S})

## Fig. 2 - $P_e$ with block-bootstrap error bars, and RH, versus $r_v$

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.repro import load_summaries
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)

rv = np.array([s['rv'] for s in S])
Pe = np.array([s['Pe'] for s in S]); lo = np.array([s['Pe_boot']['p05'] for s in S]); hi = np.array([s['Pe_boot']['p95'] for s in S])
RH = np.array([s['RH'] for s in S])
PAPER_PE = {0: 34.5, 2000: 19.2}; PAPER_RH = {0: 75.2, 200: 68.5, 500: 61.3, 1000: 53.1, 2000: 44.3}

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
a.plot(rv, Pe, 'k--', lw=.5); a.errorbar(rv, Pe, yerr=[Pe - lo, hi - Pe], fmt='ko', ms=5)
b.plot(rv, RH, 'k--', lw=.5); b.plot(rv, RH, 'ko', ms=5)
a.plot(list(PAPER_PE), list(PAPER_PE.values()), 'x', c='tab:red', label='paper')
b.plot(list(PAPER_RH), list(PAPER_RH.values()), 'x', c='tab:red', label='paper')
a.set_xlabel('$r_v$ (s m$^{-1}$)'); a.set_ylabel('$P_e$ (mm hr$^{-1}$)'); a.legend(frameon=False)
b.set_xlabel('$r_v$ (s m$^{-1}$)'); b.set_ylabel('RH (%)')
for ax in (a, b): ax.spines[['top', 'right']].set_visible(False)
a.set_title('(a)', loc='left', fontweight='bold'); b.set_title('(b)', loc='left', fontweight='bold')
plt.show()

print('P_e (mm/hr) [5th-95th bootstrap]:', ', '.join(f'{r:g}: {p:.1f} [{l:.1f}-{h:.1f}]' for r, p, l, h in zip(rv, Pe, lo, hi)))
print('paper: P_e 34.5 (rv=0) -> 19.2 (rv=2000) mm/hr, a 57 % decrease')
print(f'here:  P_e {Pe[0]:.1f} -> {Pe[-1]:.1f} mm/hr, a {100 * (1 - Pe[-1] / Pe[0]):.0f} % decrease; RH {RH[0]:.1f} -> {RH[-1]:.1f} %')

## Fig. 3 - decomposition of the $P_e$ scaling rate with RH
Fractional changes between adjacent $r_v$ simulations (Eq. 4-5), summed and divided by $\delta\ln$RH. Values are "% per %". Error bars: 5th-95th percentile over the 100 block-bootstrap replicates (paired $P_e$ / $C_e$ draws).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.repro import load_summaries, decompose
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)

pt = decompose(S)['total']
boot = decompose([dict(rv=s['rv'], RH=s['RH'], z=s['z'], Pe=s['Pe_boot']['dist'], Ce=s['Ce_boot'],
                       dy=s['dy_boot'], th=s['th_boot']) for s in S])['total']
PAPER = dict(P=1.1, eff=0.8, C=0.3, dynamic=0.23, thermo=0.06)
labels = dict(P='Precipitation $P_e$', eff='Efficiency $\\epsilon_p$', C='Condensation $C_e$', dynamic='Dynamic', thermo='Thermodynamic')

fig, ax = plt.subplots(figsize=(7, 3.2))
for i, k in enumerate(['P', 'eff', 'C', 'dynamic', 'thermo']):
    y = 4 - i
    lo, hi = np.percentile(boot[k], [5, 95])
    ax.plot([lo, hi], [y, y], 'k'); ax.plot(pt[k], y, 'ko')
    ax.plot(PAPER[k], y, 'x', c='tab:red', label='paper' if i == 0 else None)
    ax.text(min(lo, 0) - .03, y, labels[k], ha='right', va='center', fontsize=10)
ax.axvline(0, c='.7', lw=.5); ax.set_yticks([]); ax.set_xlabel('scaling rate with RH (% per %)')
ax.spines[['left', 'top', 'right']].set_visible(False); ax.legend(frameon=False, loc='lower right')
plt.show()

print(f"{'':14s}{'SAM_HEAT':>10s}{'[5-95]':>16s}{'paper':>8s}")
for k in ['P', 'eff', 'C', 'dynamic', 'thermo', 'residual']:
    lo, hi = np.percentile(boot[k], [5, 95])
    print(f"{k:14s}{float(pt[k]):10.2f}  [{lo:5.2f},{hi:5.2f}]{PAPER.get(k, float('nan')):8.2f}")
print('(residual = C - dynamic - thermo: mean-of-products vs product-of-means mismatch; not in the paper)')
d = decompose(S)
print('\nper adjacent pair (rv pairs', [f'{a:g}->{b:g}' for a, b in zip(d['rv'][:-1], d['rv'][1:])], '):')
for k in ['P', 'eff', 'C', 'dynamic', 'thermo']:
    print(f'  {k:8s}', np.round(d['pairs'][k], 2))

## Diagnostics: $C_e$, efficiency and LCL by run

In [ ]:
import pandas as pd
from samheat.analysis.repro import load_summaries
S = load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)
df = pd.DataFrame([dict(rv=s['rv'], RH=s['RH'], LCL_m=s['zlcl'], Pe=s['Pe'], Ce=s['Ce'], eff_pct=100 * s['Pe'] / s['Ce'],
                        mean_C=s['mean_C']) for s in S]).set_index('rv').round(2)
display(df)